In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
train = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_01/Data/train_features.pkl')

In [18]:
cutoff_date = train['date'].max() - pd.Timedelta(days = 90)
train_set = train[train['date'] < cutoff_date]
test_set = train[train['date'] >= cutoff_date]

print(train_set.shape, test_set.shape)

(2838726, 15) (162162, 15)


In [17]:
# printing weekdays just to see what is there
date1 = train_set['date'].max()
date2 = test_set['date'].min()
print(date1.isoweekday())
print(date2.isoweekday())

2
3


In [14]:
# checking if my sales_lag_7 data breaking or not on splitting
sample = train[(train['store_nbr']==1) & (train['family']=='AUTOMOTIVE')].sort_values('date')
sample[(sample['date'] >= cutoff_date - pd.Timedelta(days=3)) & (sample['date'] <= cutoff_date + pd.Timedelta(days=3))][['date','sales','sales_lag_7']]

,date,sales,sales_lag_7
2833380,2017-05-14,2.0,3.0
2835162,2017-05-15,9.0,5.0
2836944,2017-05-16,3.0,2.0
2838726,2017-05-17,2.0,2.0
2840508,2017-05-18,4.0,4.0
2842290,2017-05-19,5.0,4.0
2844072,2017-05-20,4.0,2.0


In [20]:
test_set = test_set.copy()
test_set['naive_pred'] = test_set['sales_lag_1']
#moving average baseline
test_set['ma_pred'] = test_set['rolling_mean_7']

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# dropping un-necessary rows(with NaN's)
eval_set = test_set.dropna(subset = ['sales', 'naive_pred', 'ma_pred'])

# naive mse and mae
naive_mae = mean_absolute_error(eval_set['sales'], eval_set['naive_pred'])
naive_mse = np.sqrt(mean_squared_error(eval_set['sales'], eval_set['naive_pred']))

# moving average mse and mae
ma_mae = mean_absolute_error(eval_set['sales'], eval_set['ma_pred'])
ma_mse = np.sqrt(mean_squared_error(eval_set['sales'], eval_set['ma_pred']))

print("naive baseline: ", '\n', "rmse: ", naive_mse, "mae: ", naive_mae)
print("moving average baseline: ", '\n', "rmse: ", ma_mse, "mae: ", ma_mae)


naive baseline:  
 rmse:  469.31729743359404 mae:  128.10087739330484
moving average baseline:  
 rmse:  395.25144514484526 mae:  109.03250419366013


In [24]:
train_set.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_01/Data/train_set.pkl')
test_set.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_01/Data/test_set.pkl')